In [1]:
import numpy as np
import pandas as pd
import json

print("Loading dataset and dependencies...")

# 1. Load the main congestion dataframe
master = pd.read_csv('master_congestion_scored.csv') # or 'master_congestion_final.csv'

# 2. Load the numpy arrays
locations = np.load('locations_list.npy', allow_pickle=True)
gnn_predictions = np.load('gnn_predictions.npy')
gnn_attention = np.load('gnn_attention.npy')
segment_mask = np.load('segment_mask.npy')

# 3. location_groups is expected to be a dictionary.
# If you didn't save it as a JSON, you may need to recreate it.
# As an example, if your 'source' format is "A.Zhandong Road1.csv", "A.Zhandong Road2.csv"
location_groups = {}
for loc in locations:
    # Just an example of how to reconstruct location groups based on your strings
    # Modify this if your 'location_groups' was built differently!
    location_groups[loc] = master[master['source'].str.contains(loc)]['source'].unique().tolist()

print("Data loaded successfully!")


Loading dataset and dependencies...
Data loaded successfully!


In [3]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB"
      if torch.cuda.is_available() else "")

True
Tesla T4
VRAM: 15.6 GB


In [2]:
def build_traffic_context(location_idx, locations, location_groups,
                           master, gnn_predictions, gnn_attention,
                           segment_mask):
    """
    Build structured natural language context for one location.
    This is the grounding layer — LLM never hallucinates numbers
    because everything is explicitly provided.
    """
    loc  = locations[location_idx]
    segs = location_groups[loc]
    pred = gnn_predictions[location_idx]   # (30,) km/h
    attn = gnn_attention[location_idx]     # (max_segments,)

    # ── Current state from most recent segment ────────────
    last_seg = segs[-1]
    subset   = master[master['source'] == last_seg].sort_values('frame_num')
    recent   = subset.tail(100)

    current_speed    = recent['avg_speed'].iloc[-1]
    current_vehicles = recent['vehicle_count'].iloc[-1]
    current_flow     = recent['total_flow'].iloc[-1]
    current_stopped  = recent['pct_stopped'].iloc[-1] * 100

    # ── Trend over last 20 frames ─────────────────────────
    trend_val  = recent['avg_speed'].tail(20).diff().mean()
    trend_word = ("worsening" if trend_val < -0.05 else
                  "improving" if trend_val > 0.05  else "stable")

    # ── Historical baseline ───────────────────────────────
    all_loc_data = master[master['source'].isin(segs)]
    hist_mean    = all_loc_data['avg_speed'].mean()
    hist_std     = all_loc_data['avg_speed'].std()
    hist_min     = all_loc_data['avg_speed'].min()
    hist_max     = all_loc_data['avg_speed'].max()

    vs_baseline = ((current_speed - hist_mean) / hist_mean * 100)
    severity    = ("critical"  if current_speed < hist_mean - 1.5*hist_std else
                   "severe"    if current_speed < hist_mean - hist_std     else
                   "moderate"  if current_speed < hist_mean                else
                   "light"     if current_speed < hist_mean + hist_std     else
                   "free flow")

    # ── Forecast summary ──────────────────────────────────
    pred_mean  = pred.mean()
    pred_min   = pred.min()
    pred_max   = pred.max()
    pred_trend = ("worsening" if pred[-1] < pred[0] - 0.3 else
                  "improving" if pred[-1] > pred[0] + 0.3 else "stable")
    pred_trajectory = [round(float(v), 2) for v in pred[:10]]

    # ── Dominant segment from attention ──────────────────
    real_attn   = attn[:len(segs)]
    dom_idx     = int(np.argmax(real_attn))
    dom_seg     = segs[dom_idx]
    dom_weight  = float(real_attn[dom_idx])

    # Build context string
    context = f"""
LOCATION: {loc} (Hohhot, Inner Mongolia, China)
RECORDING SEGMENTS ANALYZED: {len(segs)} ({', '.join(segs)})

CURRENT TRAFFIC STATE (most recent observation):
- Average speed: {current_speed:.2f} km/h
- Vehicle count: {int(current_vehicles)} vehicles in scene
- Total directional flow: {int(current_flow)} vehicles
- Percentage stopped: {current_stopped:.1f}%
- Short-term trend (last 20 frames): {trend_word}
- Congestion severity: {severity}
- vs. historical baseline: {vs_baseline:+.1f}% {'below' if vs_baseline < 0 else 'above'} normal

HISTORICAL CONTEXT FOR THIS LOCATION:
- Typical average speed: {hist_mean:.2f} km/h (±{hist_std:.2f})
- Observed range: {hist_min:.2f} – {hist_max:.2f} km/h
- Most informative recording session: {dom_seg} (attention weight: {dom_weight:.3f})
- This means: {dom_seg} best represents this location's typical traffic behavior

GNN FORECAST (next 30 frames):
- Predicted speed trajectory: {pred_trend}
- Predicted mean speed: {pred_mean:.2f} km/h
- Predicted range: {pred_min:.2f} – {pred_max:.2f} km/h
- First 10 predicted values (km/h): {pred_trajectory}
"""
    return context, severity, pred_trend


# Test the context builder on one location
ctx, sev, trend = build_traffic_context(
    0, locations, location_groups,
    master, gnn_predictions, gnn_attention, segment_mask
)
print(ctx)


LOCATION: A.Zhandong (Hohhot, Inner Mongolia, China)
RECORDING SEGMENTS ANALYZED: 2 (A.Zhandong Road1, A.Zhandong Road2)

CURRENT TRAFFIC STATE (most recent observation):
- Average speed: 6.76 km/h
- Vehicle count: 101 vehicles in scene
- Total directional flow: 72 vehicles
- Percentage stopped: 55.4%
- Short-term trend (last 20 frames): stable
- Congestion severity: moderate
- vs. historical baseline: -31.9% below normal

HISTORICAL CONTEXT FOR THIS LOCATION:
- Typical average speed: 9.94 km/h (±5.00)
- Observed range: 2.67 – 44.28 km/h
- Most informative recording session: A.Zhandong Road2 (attention weight: 0.568)
- This means: A.Zhandong Road2 best represents this location's typical traffic behavior

GNN FORECAST (next 30 frames):
- Predicted speed trajectory: stable
- Predicted mean speed: 8.20 km/h
- Predicted range: 7.94 – 8.53 km/h
- First 10 predicted values (km/h): [8.11, 8.26, 8.13, 8.05, 7.94, 8.03, 8.05, 8.21, 8.25, 8.03]



In [3]:
!pip install -q transformers accelerate "bitsandbytes>=0.46.1" qwen-vl-utils

In [4]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from PIL import Image
import torch
import numpy as np

# ── Load model ────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen2-VL-7B-Instruct"

print("Loading Qwen2-VL-7B...")
processor = AutoProcessor.from_pretrained(MODEL_NAME)

# 4-bit quantization config
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto"
)
print("Model loaded with 4-bit quantization")


def query_qwen_text_only(context, question=None):
    """
    Query Qwen2-VL with text context only (no image).
    Drop-in replacement for query_llm() from before.
    """
    user_content = f"""You are an expert traffic management AI for
Hohhot, Inner Mongolia. Analyze this traffic data:

{context}

{"Question: " + question if question else
 "Provide a complete traffic situation analysis."}"""

    messages = [{"role": "user", "content": user_content}]
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(text=[text], return_tensors="pt").to(model.device)

    print("Generating text-only response...")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True
        )

    generated = output_ids[:, inputs.input_ids.shape[1]:]
    response = processor.batch_decode(generated, skip_special_tokens=True)[0]

    # Explicitly clear GPU cache and release memory
    del inputs, output_ids, generated
    torch.cuda.empty_cache()
    print("Text-only response generated. Cache cleared and memory released.")

    return response


def query_qwen_with_frame(context, image_path, question=None):
    """
    Query Qwen2-VL with CCTV frame + quantitative context.
    This is the multimodal capability that makes Qwen2-VL special.
    """
    image = Image.open(image_path).convert("RGB")

    user_content = [
        {"type": "image",  "image": image},
        {"type": "text",   "text": f"""You are an expert traffic
management AI for Hohhot, Inner Mongolia.

Look at this CCTV frame and analyze it together with the
quantitative pipeline data below:

{context}

{"Question: " + question if question else
 "Describe what you see in the frame and how it relates "
 "to the quantitative data. Identify causes of congestion "
 "visible in the image and provide recommendations."}"""}
    ]

    messages = [{"role": "user", "content": user_content}]
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[text],
        images=[image],
        return_tensors="pt"
    ).to(model.device)

    print(f"Generating response with image from {image_path}...")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True
        )

    generated = output_ids[:, inputs.input_ids.shape[1]:]
    response = processor.batch_decode(generated, skip_special_tokens=True)[0]

    # Explicitly clear GPU cache and release memory
    del inputs, output_ids, generated
    del image # Also delete the PIL Image object
    torch.cuda.empty_cache()
    print("Image-based response generated. Cache cleared and memory released.")

    return response


# ── Test text-only first ──────────────────────────────────
ctx, severity, pred_trend = build_traffic_context(
    0, locations, location_groups,
    master, gnn_predictions, gnn_attention, segment_mask
)

print("Testing Qwen2-VL text reasoning...")
response = query_qwen_text_only(ctx)
print(response)


Loading Qwen2-VL-7B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

Model loaded with 4-bit quantization
Testing Qwen2-VL text reasoning...
Generating text-only response...
Text-only response generated. Cache cleared and memory released.
The current traffic situation at A.Zhandong Road in Hohhot, Inner Mongolia is characterized by moderate congestion. The average speed of vehicles is 6.76 km/h, which is significantly lower than the typical average speed of 9.94 km/h. The vehicle count in the scene is 101 vehicles, with a total directional flow of 72 vehicles. The percentage of vehicles stopped is 55.4%, indicating that a significant portion of vehicles are not moving. The short-term trend over the last 20 frames is stable, suggesting that the current congestion level is likely to persist in the near future.
The GNN forecast predicts that the speed trajectory will remain stable over the next 30 frames, with a predicted mean speed of 8.20 km/h and a predicted range of 7.94 – 8.53 km/h. The first 10 predicted values range from 8.11 km/h to 8.25 km/h.
In s

In [8]:
# Extract a frame from your video at a specific timestamp
def extract_video_frame(video_path, frame_number, output_path):
    import cv2
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
    ret, frame = cap.read()
    if ret:
        cv2.imwrite(output_path, frame)
    cap.release()
    return output_path

# Then query with both vision and quantitative data
frame_path = extract_video_frame(
    'hohhot_traffic.mp4',
    frame_number=3,            # the anomaly frame we found earlier
    output_path='frame_3.jpg'
)

response = query_qwen_with_frame(
    context=ctx,
    image_path=frame_path,
    question="Tell me about the traffic"
)
print(response)

Generating response with image from frame_3.jpg...
Image-based response generated. Cache cleared and memory released.
The traffic in the image appears to be moderate, with an average speed of 6.76 km/h and a total of 101 vehicles in the scene. The percentage of vehicles stopped is 55.4%, and the short-term trend over the last 20 frames is stable. The congestion severity is moderate, and the traffic is 31.9% below the historical baseline.

The GNN forecast predicts a stable speed trajectory with a mean speed of 8.20 km/h and a range of 7.94 to 8.53 km/h over the next 30 frames. The first 10 predicted values are as follows: [8.11, 8.26, 8.13, 8.05, 7.94, 8.03, 8.05, 8.21, 8.25, 8.03].

Overall, the traffic appears to be moderate with some vehicles stopped and a stable trend over the last 20 frames.
